# 05 · Measure connection timing

Measure idle API calls, including OS port opening and protocol handshake. No monitoring is started; these timings do not measure shutdown during active I/O.

Run top to bottom in a **Python 3.12+** kernel. This notebook includes all configuration and setup; no sibling example files are imported. Run only one hardware notebook/process at a time.

**Prepare the filled cooling loop before connecting.** T255P connection sends U, takes serial control and can change AUTO START to RUN, even with target writes disabled. Software disconnect does not stop the pump/chiller or restore its target. Physical hardware has not been validated with this package.

## 1. Environment and imports

Select a kernel with this package and pyserial installed. From the repository root install with `%pip install -e .`; from the examples directory use `%pip install -e ..`. The script-only dashboard (04_dashboard.py) needs the `[gui]` extra, e.g. `%pip install -e "..[gui]"`. Restart the kernel after installation or driver source changes. Imports open no serial ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from time import perf_counter

from tark_chiller.controller import T255P, TarkChiller

## 2. Choose the port

This example connects to a **ThermoTek T255P**. Set `PORT` to its serial port (for example `"COM9"`). Adapter metadata identifies the USB adapter, not the attached chiller.

The connection cell below spells out the T255P profile: 9600 baud, 8N1, XON/XOFF, no hardware flow control, and RTS/DTR deasserted. Check that your adapter/cable supports these line levels. Distilled-water target bounds are 5–45 °C, in 0.1 °C steps. See the [hardware guide](../docs/hardware.md) for wiring and the separate CAL protocol.

In [5]:
PORT = "COM9"  # Set the identified serial port, e.g. "COM9".

## 3. Connection settings

The next operation cell constructs the driver directly with the settings shown. Construction opens nothing. Its context connects and releases the port, including on a read error or interrupt. Among the notebooks, only example 02 enables target writes; dashboard script 04 also enables explicit GUI target requests.

## 4. Measure connection timing

Measure idle API calls, including OS port opening and protocol handshake. No monitoring is started; these timings do not measure shutdown during active I/O.

Run the next cell as a unit: it owns connection and cleanup together. Interrupt the cell to cancel; errors remain visible.

In [6]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=False),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

print(f"Port: {chiller.settings['port']}; transaction timeout: {chiller.timeout_s:g} s")
assert not chiller.is_connected

connect_ok = disconnect_ok = False
try:
    started = perf_counter()
    try:
        chiller.connect()
        connect_ok = True
    finally:
        connect_ms = (perf_counter() - started) * 1000
        print(f'connect():    {connect_ms:10.3f} ms | success={connect_ok}')
    assert chiller.is_connected
finally:
    started = perf_counter()
    try:
        chiller.disconnect()
        disconnect_ok = True
    finally:
        disconnect_ms = (perf_counter() - started) * 1000
        print(f'disconnect(): {disconnect_ms:10.3f} ms | success={disconnect_ok}')

assert not chiller.is_connected
print('Connection cycle complete; serial port released.')

Port: COM9; transaction timeout: 3 s
connect():        33.266 ms | success=True
disconnect():    127.934 ms | success=True
Connection cycle complete; serial port released.


## 5. Check results and finish

Each call prints elapsed milliseconds and success. Rerun the operation cell for another manually initiated cycle; idle timings are not physical timing guarantees.

Context cleanup closes software resources, including after a read error or kernel interrupt. If cleanup fails, resources may remain owned: allow pending I/O to finish and retry `chiller.disconnect(timeout_s=5.0)` if `chiller` was created. Do not rerun configuration over a live connection. A forced kernel termination cannot guarantee cleanup. Follow the equipment's physical shutdown procedure separately.